# Exercise 3 · Let Codex write the reader, then check it yourself

Your tests decide if Codex's reader is right.

**Goal:** Name the test that caught each fault in Codex's first reader, then have Codex fix it.

**Why it matters:** An agent's code can look right and still be wrong.

**Before you start:** Step 1 writes the files that exercise 2 ends with.

**What the check measures:** the reader passes your unchanged tests and six more returns. `review.json` is right. You committed all of it.

**Done when:** the last cell prints `Done:`.

## The situation

`return_reader.py` has no body. `drafts/codex_first_reader.py` is Codex's first answer (simulated), with three faults. The files are made up.

## Step 1 · Write the practice files

This is the solved copy. Run the cell. It writes `practice/` as it is after Step 6: the review, the fixed reader, and three commits. It never overwrites a file.

Expect nine `Wrote practice/` lines and `Committed the start, the review and the fixed reader`.

In [ ]:
# Step 1: write the practice files, keeping any that already exist
from pathlib import Path

READER = '''import xml.etree.ElementTree as ET

NS = {"efile": "http://www.irs.gov/efile"}
WAGES = "efile:ReturnData/efile:IRS1040/efile:WagesAmt"
NAMES = "efile:ReturnData/efile:IRS1040/efile:DependentDetail/efile:DependentFirstNm"


def read_values(xml_path: str) -> dict[str, str | list[str] | None]:
    """Read the wages and the dependents' names from one return.

    Args:
        xml_path: The return's XML file.

    Returns:
        "wages": WagesAmt's text, "" if empty, None if absent.
        "dependents": each DependentFirstNm's text, in document order.

    Raises:
        xml.etree.ElementTree.ParseError: The file is not well-formed XML.
    """
'''
GOOD = """<!-- Every value in this file is made up. -->
<Return xmlns="http://www.irs.gov/efile">
  <ReturnHeader>
    <PrimarySSN>400001234</PrimarySSN>
  </ReturnHeader>
  <ReturnData>
    <IRS1040 documentId="DOC0001">
      <WagesAmt>41000</WagesAmt>
      <DependentDetail>
        <DependentFirstNm>TEST CHILD ONE</DependentFirstNm>
      </DependentDetail>
      <DependentDetail>
        <DependentFirstNm>TEST CHILD TWO</DependentFirstNm>
      </DependentDetail>
    </IRS1040>
  </ReturnData>
</Return>
"""
LINES = GOOD.splitlines(keepends=True)
FILES = {
    "return_reader.py": READER + '    raise NotImplementedError("Exercise 3 writes this")\n',
    "fixtures/good.xml": GOOD,
    "fixtures/no_dependents.xml": "".join(line for line in LINES if "Dependent" not in line),
    "fixtures/no_wages.xml": "".join(line for line in LINES if "WagesAmt" not in line),
    "fixtures/cut_off.xml": "".join(LINES[:10]),
    "test_return_reader.py": """import xml.etree.ElementTree as ET

import pytest

from return_reader import read_values


@pytest.mark.parametrize("name, expected", [
    ("wages", "41000"),
    ("dependents", ["TEST CHILD ONE", "TEST CHILD TWO"]),
], ids=["wages", "dependents"])
def test_good_return(name: str, expected: str | list[str]) -> None:
    assert read_values("fixtures/good.xml")[name] == expected


def test_no_dependents_is_empty_list() -> None:
    assert read_values("fixtures/no_dependents.xml")["dependents"] == []


def test_missing_wages_is_absent() -> None:
    assert read_values("fixtures/no_wages.xml")["wages"] is None


def test_cut_off_file_is_an_error() -> None:
    with pytest.raises(ET.ParseError):
        read_values("fixtures/cut_off.xml")
""",
    "drafts/codex_first_reader.py": "# Codex's first answer (simulated), for Step 3.\n" + READER + '''    try:
        root = ET.parse(xml_path).getroot()
    except ET.ParseError:
        return {"wages": None, "dependents": []}
    wages = root.find(WAGES, NS)
    first = root.find(NAMES, NS)
    return {"wages": wages.text if wages else None, "dependents": [first.text] if first is not None else []}
''',
    "review.json": """{
  "except ET.ParseError:": "test_cut_off_file_is_an_error",
  "wages.text if wages else None": "test_good_return[wages]",
  "[first.text] if first is not None else []": "test_good_return[dependents]"
}
""",
    ".gitignore": "__pycache__/\n",
}

PRACTICE = Path("practice")
for name, text in FILES.items():
    path = PRACTICE / name
    if path.exists():
        print(f"Kept practice/{name}: it is already there.")
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(text, encoding="utf-8")
        print(f"Wrote practice/{name}")

# The review a learner writes in Step 4, and the reader Codex may leave in Step 5 (simulated), each committed.
import subprocess

EMPTY_REVIEW = """{
  "except ET.ParseError:": "",
  "wages.text if wages else None": "",
  "[first.text] if first is not None else []": ""
}
"""
FIXED = READER + """    root = ET.parse(xml_path).getroot()
    names = [element.text for element in root.findall(NAMES, NS)]
    return {"wages": root.findtext(WAGES, namespaces=NS), "dependents": names}
"""
git = ["git", "-C", str(PRACTICE)]
if not (PRACTICE / ".git").exists():
    (PRACTICE / "review.json").write_text(EMPTY_REVIEW, encoding="utf-8")
    for step in (["init", "-q", "-b", "main"], ["add", "-A"], ["commit", "-qm", "Start"]):
        subprocess.run(git + step, check=True)
    (PRACTICE / "return_reader.py").write_text(FILES["drafts/codex_first_reader.py"], encoding="utf-8")
    (PRACTICE / "review.json").write_text(FILES["review.json"], encoding="utf-8")
    subprocess.run(git + ["commit", "-qam", "Review Codex's first reader"], check=True)
    (PRACTICE / "return_reader.py").write_text(FIXED, encoding="utf-8")
    subprocess.run(git + ["commit", "-qam", "Read the return with ElementTree"], check=True)
    print("Committed the start, the review and the fixed reader")

## Step 2 · Open a terminal and commit the start

Choose **Terminal > New Terminal**. Type each line, then press Enter:

```sh
source .venv/bin/activate
cd week-02/day-04/exercise-03-codex-writes-the-reader/practice
git init -q -b main && git add -A && git commit -qm "Start"
```

Expect no output from the last line.

## Step 3 · Run your tests on Codex's first answer

```sh
cp drafts/codex_first_reader.py return_reader.py
python -m pytest -q
```

Expect `3 failed, 2 passed, 3 warnings`. Each `FAILED` line names a test. The warnings are about `if wages`.

## Step 4 · Write the review

Open it with `code review.json`. Its keys are the three pieces of code in `return_reader.py` that hold a fault. For each one, write the test that failed on it, as its `FAILED` line names it. Save the file. Commit all, so Step 5's diff shows only Codex's fix:

```sh
git commit -qam "Review Codex's first reader"
```

<details><summary>Show the answer</summary>

```json
{
  "except ET.ParseError:": "test_cut_off_file_is_an_error",
  "wages.text if wages else None": "test_good_return[wages]",
  "[first.text] if first is not None else []": "test_good_return[dependents]"
}
```

Why: the cut-off test expects `ParseError`, but the `except` returns a result. The wages test gets `None`: "Caution: Elements with no subelements will test as False." ([ElementTree](https://docs.python.org/3/library/xml.etree.elementtree.html)) The dependents test gets one name, as `find` returns the first only.

</details>

## Step 5 · Ask Codex to fix the reader

```sh
codex --no-daemon
```

If Codex stops with an error about `--no-daemon`, run `codex` without it. If Codex asks "Trust this folder?", choose `1. Trust and continue`. Type this request:

```text
Make all tests in test_return_reader.py pass. Use only xml.etree.ElementTree. Do not edit test_return_reader.py or fixtures/. Test for absence with is None. Follow the docstring of read_values. Done when python -m pytest -q shows 5 passed.
```

Type `/diff`. Read the change. Press `q`. Type `/exit`.

Expect a diff in `return_reader.py` only.

## Step 6 · Run the tests yourself, then commit

```sh
python -m pytest -q
git diff --stat
```

Expect `5 passed`, then `1 file changed` for `return_reader.py`. If a test or fixture changed, run `git restore test_return_reader.py fixtures`, and go back to Step 5. Then commit:

```sh
git commit -qam "Read the return with ElementTree"
```

## Start over (only if you need it)

Set `START_OVER = True` and run the cell. It writes the Step 1 files again, with an empty `review.json`. Then set it back to `False` and go to Step 3.

Expect `Nothing changed.` while it is `False`.

In [ ]:
# Start over: only if START_OVER is True, write the Step 1 files again
START_OVER = False

if not START_OVER:
    print("Nothing changed.")
elif "FILES" not in globals():
    print("Run the Step 1 cell first, then this cell.")
else:
    for name, text in FILES.items():
        (PRACTICE / name).parent.mkdir(parents=True, exist_ok=True)
        (PRACTICE / name).write_text(text, encoding="utf-8")
    print("Wrote the practice files again. Now set START_OVER back to False.")

## Check your work

Run the cell. It also tries six returns. One has an empty `<WagesAmt></WagesAmt>`, which no test of yours holds: `.text` gives `None`, and the docstring says `""`. Expect six `ok:` lines, then `Done:`. A `Not yet:` line names the first problem. Paste it into Codex if it names the reader.

In [ ]:
# Check your work: the imports, Git, your tests and six more returns
import json
import re
import runpy
import shutil
import subprocess
import sys
import tempfile
import xml.etree.ElementTree as ET
from pathlib import Path

PRACTICE = Path("practice")
CAUGHT = ['test_cut_off_file_is_an_error', 'test_good_return[wages]', 'test_good_return[dependents]']
KIDS = "".join(f"<DependentDetail><DependentFirstNm>{n}</DependentFirstNm></DependentDetail>" for n in "ABC")
MORE = [("<WagesAmt>52000</WagesAmt>", "wages", "52000"), ("", "wages", None), ("<WagesAmt></WagesAmt>", "wages", ""),
        (KIDS, "dependents", ["A", "B", "C"]), ("", "dependents", []), (KIDS, "cut off", "ParseError")]


def git(*args):
    return subprocess.run(["git", "-C", str(PRACTICE), *args], capture_output=True, text=True).stdout.strip()


def reads_wrong(read_values, folder):
    for inner, key, want in MORE:
        text = f'<Return xmlns="http://www.irs.gov/efile"><ReturnData><IRS1040>{inner}</IRS1040></ReturnData></Return>'
        (folder / "more.xml").write_text(text[:80] if key == "cut off" else text, encoding="utf-8")
        try:
            got = read_values(str(folder / "more.xml")).get(key, "a result")
        except ET.ParseError:
            got = "ParseError"
        if got != want:
            return f"read_values gives {key} {got!r}, not {want!r}, for <IRS1040>{inner[:40]} (Step 5)."
    return ""


def first_problem():
    """Print ok for each check that passes; return the first problem, or ""."""
    code = (PRACTICE / "return_reader.py").read_text(encoding="utf-8")
    outside = [m for m in re.findall(r"^\s*(?:from|import) (\w+)", code, re.M) if m not in sys.stdlib_module_names]
    if outside:
        return f"return_reader.py imports {outside[0]}, which Python does not include (Step 5)."
    print("ok: the reader uses the standard library only")
    if not (PRACTICE / ".git").exists():
        return "practice/ has no Git repository. Do Step 2."
    start = git("rev-list", "--max-parents=0", "HEAD")
    if not start:
        return "practice/ has no commit. Commit the start (Step 2)."
    if git("diff", "--name-only", start, "--", "test_return_reader.py", "fixtures"):
        return "a test or fixture changed. A test moved to meet the code proves nothing. Undo it (Step 6)."
    print("ok: the tests and fixtures did not change")
    with tempfile.TemporaryDirectory() as tmp:
        work = shutil.copytree(PRACTICE, Path(tmp) / "p", ignore=shutil.ignore_patterns(".git", "__pycache__"))
        run = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider"], cwd=work,
                             capture_output=True, text=True, timeout=300)
        if run.returncode:
            return f"pytest says {run.stdout.strip().splitlines()[-1]}. Ask Codex again (Step 5)."
        print("ok: your 5 tests pass")
        wrong = reads_wrong(runpy.run_path(str(work / "return_reader.py"))["read_values"], work)
    if wrong:
        return wrong
    print("ok: it reads six more returns right")
    review = json.loads((PRACTICE / "review.json").read_text(encoding="utf-8"))
    if [str(test).strip().removeprefix("test_return_reader.py::") for test in review.values()] != CAUGHT:
        return "review.json names a wrong test. Read the FAILED lines again (Step 4)."
    print("ok: review.json is right")
    if len(git("log", "--format=%s").splitlines()) < 3 or git("status", "--porcelain", "return_reader.py", "review.json"):
        return "the review or the reader is not committed (Step 4 or Step 6)."
    print("ok: all is committed")
    return ""


try:
    problem = first_problem()
except Exception as error:
    problem = f"the check stopped on {type(error).__name__}: {error}. Run Step 1, then python -m pytest -q (Step 3)."
print(f"Not yet: {problem}" if problem else "Done: the reader passes your tests and six more returns. Your review is right.")

## What you learned

- Your tests and your own run decide if the agent's code is right.

**Reflect:** How would you notice an agent that edits a test to pass it?